# Mask erosion loss analysis
Reproducible notebook to generate:
- mask_erosion_loss_report.csv
- mask_erosion_boundary_white_report.csv

In [5]:
import os
from pathlib import Path

import cv2
import numpy as np
import pandas as pd

# paths are relative to the analysis folder
masks_dir = Path('..') / 'data' / 'PANCREAS_PREPROCESSED'
loss_out = Path('mask_erosion_loss_report.csv')
boundary_out = Path('mask_erosion_boundary_white_report.csv')

kernel_sizes = [5, 9, 10]
white_threshold = 245

mask_paths = sorted(masks_dir.glob('*_mask.png'))
print('masks_dir:', masks_dir.resolve())
print('mask files found:', len(mask_paths))

masks_dir: /home/daniduhnev/projects/master-thesis/data/PANCREAS_PREPROCESSED
mask files found: 134


In [6]:
loss_rows = []

for mask_path in mask_paths:
    study_id = mask_path.name.replace('_mask.png', '')

    mask_gray = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask_gray is None:
        continue

    mask_bin = (mask_gray > 0).astype(np.uint8)
    original_pixels = int(mask_bin.sum())
    if original_pixels == 0:
        continue

    for k in kernel_sizes:
        kernel = np.ones((k, k), np.uint8)
        eroded_mask = cv2.erode(mask_bin, kernel, iterations=1)
        eroded_pixels = int(eroded_mask.sum())
        lost_pixels = original_pixels - eroded_pixels
        lost_pct = 100.0 * lost_pixels / original_pixels

        loss_rows.append({
            'study_id': study_id,
            'kernel': k,
            'orig_pixels': original_pixels,
            'eroded_pixels': eroded_pixels,
            'lost_pixels': lost_pixels,
            'lost_pct': lost_pct,
        })

loss_df = pd.DataFrame(loss_rows).sort_values(['study_id', 'kernel'])
loss_df.to_csv(loss_out, index=False)

print('loss report rows:', len(loss_df))
print('saved:', loss_out.resolve())

loss report rows: 402
saved: /home/daniduhnev/projects/master-thesis/analysis/mask_erosion_loss_report.csv


In [7]:
boundary_rows = []
edge_kernel = np.ones((3, 3), np.uint8)

for mask_path in mask_paths:
    study_id = mask_path.name.replace('_mask.png', '')
    image_path = masks_dir / f'{study_id}_image.png'

    image_bgr = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    mask_gray = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if image_bgr is None or mask_gray is None:
        continue

    mask_bin = (mask_gray > 0).astype(np.uint8)

    for k in [0] + kernel_sizes:
        if k == 0:
            current_mask = mask_bin.copy()
        else:
            kernel = np.ones((k, k), np.uint8)
            current_mask = cv2.erode(mask_bin, kernel, iterations=1)

        inner = cv2.erode(current_mask, edge_kernel, iterations=1)
        boundary = (current_mask - inner).astype(np.uint8)

        white_pixels = (
            (image_bgr[:, :, 0] >= white_threshold)
            & (image_bgr[:, :, 1] >= white_threshold)
            & (image_bgr[:, :, 2] >= white_threshold)
        )

        boundary_pixels = int(boundary.sum())
        white_boundary_pixels = int((white_pixels & (boundary == 1)).sum())

        if boundary_pixels > 0:
            white_boundary_pct = 100.0 * white_boundary_pixels / boundary_pixels
        else:
            white_boundary_pct = 0.0

        boundary_rows.append({
            'study_id': study_id,
            'kernel': k,
            'boundary_pixels': boundary_pixels,
            'white_boundary_pixels': white_boundary_pixels,
            'white_boundary_pct': white_boundary_pct,
        })

boundary_df = pd.DataFrame(boundary_rows).sort_values(['study_id', 'kernel'])
boundary_df.to_csv(boundary_out, index=False)

print('boundary report rows:', len(boundary_df))
print('saved:', boundary_out.resolve())

boundary report rows: 536
saved: /home/daniduhnev/projects/master-thesis/analysis/mask_erosion_boundary_white_report.csv


In [8]:
print('summary')
print('')

print('loss by kernel')
print(
    loss_df.groupby('kernel')['lost_pct']
    .agg(['mean', 'median', 'min', 'max'])
    .round(3)
)

print('')
print('white boundary percent by kernel')
print(
    boundary_df.groupby('kernel')['white_boundary_pct']
    .agg(['mean', 'median', 'min', 'max'])
    .round(3)
)

print('')
print('top 5 highest loss cases at kernel 9')
print(
    loss_df[loss_df['kernel'] == 9]
    .sort_values('lost_pct', ascending=False)
    .head(5)[['study_id', 'orig_pixels', 'lost_pixels', 'lost_pct']]
)

print('')
print('top 5 highest white-boundary percent at kernel 9')
print(
    boundary_df[boundary_df['kernel'] == 9]
    .sort_values('white_boundary_pct', ascending=False)
    .head(5)[['study_id', 'white_boundary_pixels', 'boundary_pixels', 'white_boundary_pct']]
)

summary

loss by kernel
          mean  median     min     max
kernel                                
5        9.954   9.330   4.982  26.124
9       19.550  18.334   9.871  50.316
10      21.893  20.510  11.081  56.074

white boundary percent by kernel
          mean  median     min     max
kernel                                
0       49.051  49.719  27.495  81.503
5       25.563  26.331   8.780  37.500
9        0.433   0.000   0.000   2.889
10       0.398   0.000   0.000   2.715

top 5 highest loss cases at kernel 9
    study_id  orig_pixels  lost_pixels   lost_pct
226    28_03         2848         1433  50.316011
214    27_01         5964         2414  40.476190
115    14_01         5511         2111  38.305208
259    33_03         3585         1304  36.373780
400    56_03         8421         2947  34.995844

top 5 highest white-boundary percent at kernel 9
    study_id  white_boundary_pixels  boundary_pixels  white_boundary_pct
82     08_01                     23              796

In [10]:
# flag studies with high mask loss at kernel 9
threshold_pct = 30.0

k9 = loss_df[loss_df['kernel'] == 9].copy()
flagged = k9[k9['lost_pct'] > threshold_pct].sort_values('lost_pct', ascending=False)

print('kernel 9 studies:', len(k9))
print('threshold (%):', threshold_pct)
print('flagged studies:', len(flagged))
print('flagged percent:', round(100.0 * len(flagged) / len(k9), 2), '%')

print('')
print('kernel 9 lost_pct quantiles')
print(k9['lost_pct'].quantile([0.75, 0.80, 0.85, 0.90, 0.95]).round(3))

print('')
print('flagged studies list')
print(flagged[['study_id', 'orig_pixels', 'lost_pixels', 'lost_pct']])

flag_out = Path('mask_erosion_loss_flags_kernel9.csv')
flagged[['study_id', 'orig_pixels', 'lost_pixels', 'lost_pct']].to_csv(flag_out, index=False)
print('')
print('saved:', flag_out.resolve())

kernel 9 studies: 134
threshold (%): 30.0
flagged studies: 10
flagged percent: 7.46 %

kernel 9 lost_pct quantiles
0.75    22.362
0.80    23.428
0.85    24.740
0.90    28.068
0.95    32.553
Name: lost_pct, dtype: float64

flagged studies list
    study_id  orig_pixels  lost_pixels   lost_pct
226    28_03         2848         1433  50.316011
214    27_01         5964         2414  40.476190
115    14_01         5511         2111  38.305208
259    33_03         3585         1304  36.373780
400    56_03         8421         2947  34.995844
139    16_01         5040         1708  33.888889
61     08_01        10202         3339  32.728877
55     07_02         8078         2622  32.458529
13     01_05         7983         2536  31.767506
388    55_02         3811         1150  30.175807

saved: /home/daniduhnev/projects/master-thesis/analysis/mask_erosion_loss_flags_kernel9.csv
